# Merge Colonies Datasets

## Join Delhi+NDMC+JJC and UAC shapefiles

Tasks:
* Import both shapefiles **[DONE]**
* Project them to the same CRS **[DONE]**
* Harmonize columns **[DONE]**
* Find max USO ID for Delhi+NDMC+JJC **[DONE]**
* Create new ID for UACs **[DONE]**
* Merge UACs with Delhi+NDMC+JJC **[DONE]**
* Save shapefile **[DONE]**
* Attempt to open shapefile in QGIS **[DONE]**

In [ ]:
# import modules
import pickle
import pandas as pd
import geopandas as gpd
from pyproj import CRS

# Import both shapefiles

In [ ]:
delhi_ndmc_jjc = gpd.read_file("delhi_ndmc_jjc_corrected.shp")

In [ ]:
uac = gpd.read_file("uac_deduplicated_31july2020.shp")

# Reproject to CRS: EPSG 3857

Check CRS

In [ ]:
delhi_ndmc_jjc.crs

In [ ]:
uac.crs

Change CRS

In [ ]:
# WGS 84 / Delhi
epsg_code = 3857

# Define CRS in WKT format using EPSG code
target_projection = CRS.from_epsg(epsg_code).to_wkt()
    
# Reproject shapefiles to epsg_code
delhi_ndmc_jjc = delhi_ndmc_jjc.to_crs(target_projection)
uac = uac.to_crs(target_projection)

In [ ]:
uac.crs == delhi_ndmc_jjc.crs

# Visualize shapefiles and view GeoDataFrames

In [ ]:
delhi_ndmc_jjc.plot()

In [ ]:
uac.plot()

In [ ]:
delhi_ndmc_jjc.head()

In [ ]:
delhi_ndmc_jjc.columns

In [ ]:
uac.head()

In [ ]:
uac.columns

## Harmonize Columns
* UAC
    * Remove `map_no`, `registrati`, `fme_databa`, and `index` **[DONE]**
    * Add `USO_AREA_U` column **[DONE]**
    * Add `USO_FINAL` column with UAC class (find what Bijoy codes this as)
    * Add `HOUSETAX_C` column with code for non-NDMC (find out what Bijoy codes this as)
* Delhi/NDMC/JJC
    * Remove `Area` column
* Make sure only columns in both GeoDataFrames are `USO_AREA_U`, `HOUSETAX_C`, `USO_FINAL`, and `geometry`

### Unique values for `USO_FINAL`

In [ ]:
delhi_ndmc_jjc['USO_FINAL'].unique()

Code UACs' `USO_FINAL` as 'UAC2'

### Unique values for `HOUSETAX_C`

In [ ]:
delhi_ndmc_jjc['HOUSETAX_C'].unique()

Code UACs' `HOUSETAX_C` as None

### Remove extraneous columns from `uac`

In [ ]:
uac = uac.drop(columns=['map_no', 'registrati', 'fme_databa', 'index'])

In [ ]:
uac.head(2)

### Create new USO ID for UACs

In [ ]:
# Find the highest value of USO_Index for Delhi+NDMC+JJC data
delhi_ndmc_jjc['USO_AREA_U'].max()

In [ ]:
# Initialize USO_AREA_U column with -1
uac['USO_AREA_U'] = -1

In [ ]:
# Set beginning of USO AREA Code for JJCs to be 5001
uso_area_code = 5001

# Iterate through all UAC rows
for idx, row in uac.iterrows():
    
    # Set USO_AREA_U to code beginning at 5001
    uac.loc[idx, 'USO_AREA_U'] = uso_area_code
    
    # Increment USO_AREA_U code
    uso_area_code += 1

In [ ]:
uac.head()

In [ ]:
uac.tail()

### Create `USO_FINAL` column with category `UAC2`

In [ ]:
uac['USO_FINAL'] = 'UAC2'

In [ ]:
uac.head(2)

In [ ]:
uac.tail(2)

### Create `HOUSETAX_C` column with value `None`

In [ ]:
uac['HOUSETAX_C'] = None

In [ ]:
uac.head()

In [ ]:
uac.tail()

### Remove `Area` column from Delhi+NDMC+JJC

In [ ]:
delhi_ndmc_jjc = delhi_ndmc_jjc.drop(columns=['AREA'])

In [ ]:
delhi_ndmc_jjc.head()

### Merge UAC and Delhi+NDMC+JJC

In [ ]:
# Concatenate both GeoDataFrames (using pd.concat)
# and created combined GeoDataFrame with same crs as delhi_ndmc_jjc (EPSG 3857)
colonies = gpd.GeoDataFrame(pd.concat([delhi_ndmc_jjc, uac], ignore_index=True), crs=delhi_ndmc_jjc.crs)

In [ ]:
colonies.head()

In [ ]:
# check number of columns
len(colonies)

In [ ]:
colonies.plot()

# Save Combined Shapefile

In [ ]:
colonies.to_file("colonies_31july2020.shp")

In [ ]:
with open('colonies_31july2020.data', 'wb') as f:
    pickle.dump(colonies, f)